# LightGBM Ranker - Alpha 5D

Notebook này train mô hình **cross-sectional ranking** để chọn cổ phiếu mạnh hơn thị trường trong 5 phiên tới.

Target chính:
- `alpha_5d = return_5d_stock - return_5d_vnindex`

Model:
- `LightGBM Ranker (lambdarank)`
- group theo `trading_date`
- output là score để chọn `top-k` mỗi ngày


In [ ]:
%pip install -q lightgbm

from google.colab import drive
drive.mount('/content/drive')

import json
import os
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
# === BƯỚC 1: LOAD DATA ===
DATA_PATH = '/content/drive/MyDrive/all_stocks_train.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
df['trading_date'] = pd.to_datetime(df['trading_date'])
df = df.drop_duplicates(subset=['symbol', 'trading_date'], keep='last')
df = df.sort_values(['symbol', 'trading_date']).reset_index(drop=True)

print(f'Raw data: {len(df):,} dòng | {df["symbol"].nunique()} mã')
print(df[['symbol', 'trading_date']].head())


In [ ]:
# === BƯỚC 2: FILTER + MERGE VNINDEX ===
START_DATE = '2021-01-01'
MIN_VOLUME_SMA20 = 100000

vnindex = (
    df[df['symbol'] == 'VNINDEX'][['trading_date', 'close', 'return_5d']]
    .drop_duplicates(subset=['trading_date'], keep='last')
    .sort_values('trading_date')
    .reset_index(drop=True)
)

vnindex['vnindex_momentum_5'] = vnindex['close'].pct_change(5).clip(-0.3, 0.3)
vnindex['vnindex_momentum_20'] = vnindex['close'].pct_change(20).clip(-0.3, 0.3)
vnindex_diff = vnindex['close'].diff()
gain = vnindex_diff.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
loss = (-vnindex_diff.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
rs = gain / loss.replace(0, np.nan)
vnindex['vnindex_rsi'] = 100 - (100 / (1 + rs))
vnindex = vnindex.rename(columns={'return_5d': 'vnindex_return_5d'})

stocks = df[df['symbol'] != 'VNINDEX'].copy()
stocks = stocks[stocks['exchange'].isin(['HOSE', 'HNX'])]
stocks = stocks[stocks['volume_sma_20'] > MIN_VOLUME_SMA20]
stocks = stocks[stocks['trading_date'] >= START_DATE]

stocks = stocks.merge(vnindex, on='trading_date', how='left', validate='many_to_one')
stocks['alpha_5d'] = stocks['return_5d'] - stocks['vnindex_return_5d']

print(f'After filter: {len(stocks):,} dòng | {stocks["symbol"].nunique()} mã')
print(f'VNINDEX rows : {len(vnindex):,}')
print(vnindex[['trading_date', 'vnindex_return_5d', 'vnindex_momentum_5', 'vnindex_momentum_20', 'vnindex_rsi']].head(3))
print(f'Alpha rows    : {stocks["alpha_5d"].notna().sum():,}')


In [ ]:
# === BƯỚC 3: FEATURE SELECTION ===
# Tính toán các chỉ báo tương quan tương đối (vì vnindex_* tĩnh đã bị loại bỏ)
stocks['rel_momentum_5'] = (stocks['price_momentum_5'] - stocks['vnindex_momentum_5']).clip(-0.5, 0.5)
stocks['rel_momentum_20'] = (stocks['price_momentum_20'] - stocks['vnindex_momentum_20']).clip(-0.5, 0.5)
stocks['rsi_vs_vnindex'] = (stocks['rsi_14'] - stocks['vnindex_rsi']).clip(-100, 100)

# Giữ lại 10 features cốt lõi có ý nghĩa logic cao, loại bỏ nến và rank_pct dư thừa để chống overfit.
FEATURES = [
    'price_vs_sma5', 'price_vs_sma20',
    'rel_momentum_5', 'rel_momentum_20',
    'rsi_14', 'rsi_vs_vnindex',
    'volume_ratio', 'bb_width_norm',
    'cmf_20', 'atr_pct'
]

# Tạo target alpha_rank_pct (Thứ hạng phần trăm 5 ngày tới)
stocks = stocks.dropna(subset=['alpha_5d'])
stocks['alpha_rank_pct'] = stocks.groupby('trading_date')['alpha_5d'].rank(pct=True, method='first')

# Xử lý NaN cho features
stocks[FEATURES] = stocks.groupby('symbol')[FEATURES].ffill()
stocks = stocks.dropna(subset=FEATURES)

print(f"Số lượng features sử dụng: {len(FEATURES)}")
print(FEATURES)
print(f"Số lượng dòng khả dụng sau xử lý: {len(stocks):,}")


In [ ]:
# === BƯỚC 4: WALK-FORWARD VALIDATION (5 FOLDS) ===
# Chia dữ liệu cuốn chiếu và đo lường độ ổn định của Rank IC (Spearman correlation)
from scipy.stats import spearmanr

dates = sorted(stocks['trading_date'].unique())
n_days = len(dates)

# Fold config: 18 tháng train (360 ngày), 2 tháng test (40 ngày) cuốn chiếu chuẩn xác
fold_test_days = 40
fold_train_days = 360
folds = []

for i in range(5):
    test_end_idx = n_days - 1 - (i * fold_test_days)
    test_start_idx = test_end_idx - fold_test_days + 1
    train_end_idx = test_start_idx - 1
    train_start_idx = max(0, train_end_idx - fold_train_days + 1)
    
    folds.append({
        'fold': i + 1,
        'train_dates': dates[train_start_idx:train_end_idx + 1],
        'test_dates': dates[test_start_idx:test_end_idx + 1]
    })

folds = folds[::-1]

import lightgbm as lgb
ic_records = []

print("BẮT ĐẦU WALK-FORWARD BACKTESTING...\n")

for f in folds:
    train_fold = stocks[stocks['trading_date'].isin(f['train_dates'])].copy()
    test_fold = stocks[stocks['trading_date'].isin(f['test_dates'])].copy()
    
    X_tr, y_tr = train_fold[FEATURES], train_fold['alpha_rank_pct']
    X_te, y_te = test_fold[FEATURES], test_fold['alpha_rank_pct']
    
    model = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        learning_rate=0.03,
        n_estimators=100,
        max_depth=3,
        num_leaves=8,
        min_child_samples=100,
        subsample=0.7,
        colsample_bytree=0.5,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    model.fit(X_tr, y_tr)
    
    test_fold['pred_rank'] = model.predict(X_te)
    
    # Tính Rank IC theo từng ngày
    daily_ics = []
    for date, group in test_fold.groupby('trading_date'):
        if len(group) < 10:
            continue
        ic, _ = spearmanr(group['pred_rank'], group['alpha_5d'])
        if not np.isnan(ic):
            daily_ics.append(ic)
            
    mean_ic = np.mean(daily_ics) if daily_ics else 0
    std_ic = np.std(daily_ics) if daily_ics else 0
    ir = mean_ic / std_ic if std_ic > 0 else 0
    
    ic_records.append({
        'Fold': f['fold'],
        'Train Start': f['train_dates'][0].strftime('%Y-%m-%d'),
        'Train End': f['train_dates'][-1].strftime('%Y-%m-%d'),
        'Test Start': f['test_dates'][0].strftime('%Y-%m-%d'),
        'Test End': f['test_dates'][-1].strftime('%Y-%m-%d'),
        'Mean Rank IC': mean_ic,
        'Std Rank IC': std_ic,
        'IR': ir
    })
    
    print(f"Fold {f['fold']} | Test: {f['test_dates'][0].strftime('%Y-%m-%d')} -> {f['test_dates'][-1].strftime('%Y-%m-%d')}")
    print(f"  Mean Rank IC: {mean_ic:.4f} | Std IC: {std_ic:.4f} | IR (Information Ratio): {ir:.4f}\n")

df_ic = pd.DataFrame(ic_records)
print("="*60)
print("BÁO CÁO WALK-FORWARD VALIDATION")
print("="*60)
print(df_ic.to_string(index=False))
print(f"\nTrung bình Rank IC trên 5 Folds: {df_ic['Mean Rank IC'].mean():.4f}")
print(f"Trung bình IR (Information Ratio): {df_ic['IR'].mean():.4f}")


In [ ]:
# === BƯỚC 5: HUẤN LUYỆN TOÀN BỘ DỮ LIỆU (PRODUCTION MODEL) ===
# Dùng toàn bộ dữ liệu lịch sử trước TEST_CUTOFF để train mô hình cuối cùng.
TEST_CUTOFF = pd.Timestamp('2025-10-01')
train_full = stocks[stocks['trading_date'] < TEST_CUTOFF].copy()
test_full = stocks[stocks['trading_date'] >= TEST_CUTOFF].copy()

X_tr_full, y_tr_full = train_full[FEATURES], train_full['alpha_rank_pct']
X_te_full, y_te_full = test_full[FEATURES], test_full['alpha_rank_pct']

model_production = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    learning_rate=0.03,
    n_estimators=100,
    max_depth=3,
    num_leaves=8,
    min_child_samples=100,
    subsample=0.7,
    colsample_bytree=0.5,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model_production.fit(X_tr_full, y_tr_full)
print("[OK] Đã huấn luyện xong mô hình Production trên toàn bộ tập dữ liệu lịch sử!")


In [ ]:
# === BƯỚC 6: ĐÁNH GIÁ THỰC CHIẾN TRÊN TẬP TEST (155 NGÀY) ===
test_score = model_production.predict(X_te_full)

test_eval = test_full[['symbol', 'trading_date', 'alpha_5d', 'return_5d', 'vnindex_return_5d']].copy()
test_eval['score'] = test_score

def evaluate_topk(frame, ks=(10, 20, 50), ascending=False):
    rows = []
    for k in ks:
        picks = (
            frame.sort_values(['trading_date', 'score'], ascending=[True, ascending])
                 .groupby('trading_date', group_keys=False)
                 .head(k)
        )
        rows.append({
            'top_k': k,
            'samples': len(picks),
            'days': picks['trading_date'].nunique(),
            'avg_alpha': picks['alpha_5d'].mean(),
            'alpha_win_rate': (picks['alpha_5d'] > 0).mean(),
            'avg_stock_return': picks['return_5d'].mean(),
            'stock_win_rate': (picks['return_5d'] > 0).mean(),
        })
    return pd.DataFrame(rows)

test_topk = evaluate_topk(test_eval, ks=(10, 20, 50), ascending=False)
test_bottomk = evaluate_topk(test_eval, ks=(10, 20, 50), ascending=True)

universe_test = test_full.groupby('trading_date')[['alpha_5d', 'return_5d']].mean().mean()

print('\n' + '=' * 72)
print('LIGHTGBM REGRESSOR PRODUCTION REPORT')
print('=' * 72)
print(f'Test avg daily alpha universe  : {universe_test["alpha_5d"]:.4%}')
print(f'Test avg daily return universe : {universe_test["return_5d"]:.4%}')

print('\nTop-k long trên test (Mua mạnh):')
print(test_topk.to_string(index=False))

print('\nTop-k yếu nhất trên test (Tránh xa/Bán khống):')
print(test_bottomk.to_string(index=False))

# Tính toán Spearman Rank IC trên tập Test
test_ics = []
for date, group in test_eval.groupby('trading_date'):
    if len(group) < 10:
        continue
    ic, _ = spearmanr(group['score'], group['alpha_5d'])
    if not np.isnan(ic):
        test_ics.append(ic)
print(f"\nRank IC trên tập Test: {np.mean(test_ics):.4f}")


In [ ]:
# === BƯỚC 7: LƯU MODEL VÀ CONFIG ===
OUTPUT_DIR = '/content/drive/MyDrive/StockData/models'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = f'{OUTPUT_DIR}/lgbm_alpha_5d_ranker.txt'
META_PATH = f'{OUTPUT_DIR}/lgbm_alpha_5d_features.json'

model_production.booster_.save_model(MODEL_PATH)
with open(META_PATH, 'w', encoding='utf-8') as f:
    import json
    json.dump({
        'model_type': 'lightgbm_regressor',
        'target': 'alpha_rank_pct',
        'features': FEATURES
    }, f, indent=2, ensure_ascii=False)

print('Saved:')
print(MODEL_PATH)
print(META_PATH)
